In [ ]:
import itertools
import typing
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

from utils.corr_assoc_utils import create_corr_assoc_matrix
from utils.misc_utils import contingency_table, turn_cats_to_continuous
from utils.saving_utils import load_data

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True


# See answers to https://stackoverflow.com/questions/38783027/jupyter-notebook-display-two-pandas-tables-side-by-side
def display_side_by_side(*dfs: pd.DataFrame, tablespacing: int = 5):
    output = ""
    for df in dfs:
        output += df.style.set_table_attributes("style='display:inline; margin-right:10px;'")._repr_html_() or ""  # type: ignore
        output += tablespacing * "\xa0"
    display(HTML(output))


# def display_in_columns(df: pd.DataFrame, n_columns: int = 3):
#     display_side_by_side(
#         *(df.iloc[i * len(df) // n_columns : (i + 1) * len(df) // n_columns] for i in range(n_columns))
#     )

# Hardcoded paths, change to apply to new data
data_path = Path("data/preprocessed/data_full.pkl")
vars_info_path = Path("data/preprocessed/vars_info_full.pkl")

# Read in data
data = typing.cast(pd.DataFrame, load_data(data_path))
vars_info = typing.cast(pd.DataFrame, load_data(vars_info_path))

### search for variable information

In [2]:
vars_info[vars_info["name"].str.contains("hnb")]

,name,description,labels,group,type
42,hnb30,30-Tages-Prävalenz Tabakerhitzer,"{0: 'nein', 1: 'ja'}",nicotine,binary
43,hnb_fag_sch,Schwere der Abhängigkeit nach Fagerström (HNB),"{1: 'sehr geringe Abhängigkeit', 2: 'geringe A...",nicotine,ordinal
118,typ_hnb,30-Tage-Prävalenz Heat-Not-Burn,{},nicotine,binary


### count values of variable

In [ ]:
data["hnb_fag_sch"].value_counts(dropna=False, normalize=False)

### show how often two variables are equal and different

In [ ]:
# Check if two variables are equal (only works if both are continuous or categorical with the same labels)
vari, varj = "hnb30", "typ_hnb"

# True means vari is equal to varj, False means they are different
vars_equal = (data[vari] == data[varj]) | (data[vari].isnull() & data[varj].isnull())
vars_equal.value_counts(normalize=True)  # normalize=False for absolute values, normalize=True for percentages

In [ ]:
# Show instances where they are not equal
data.loc[~vars_equal, [vari, varj]]

In [ ]:
# Check if two categorical variables are the same
vari, varj = "alkkons", "alkrisk"
# Use normalize="index" to show percentages for each index
contingency_table(data[vari], data[varj], dropna=False, normalize=False)

### create table of highest correlations

In [ ]:
# Visualize correlation matrices
treat_cat_as_continuous: list[str] = vars_info.loc[vars_info["type"].isin(["binary", "ordinal"]), "name"].tolist()

corr_assoc_matrix, p_value_matrix = create_corr_assoc_matrix(turn_cats_to_continuous(data, treat_cat_as_continuous))
corr_assoc_table = (
    pd.DataFrame(
        [
            [vari, varj, corr_assoc, p_value_matrix.loc[vari, varj]]
            for vari, varj in itertools.combinations(vars_info["name"], r=2)
            if (corr_assoc := corr_assoc_matrix.loc[vari, varj]) > 0.9  # type: ignore
        ]
    )
    .set_axis(["vari", "varj", "corr_assoc", "p_value"], axis=1)
    .sort_values("corr_assoc", ascending=False)
)
corr_assoc_table

In [ ]:
# Correlations excluding some variables
excluded_vars = [
    # alcohol
    "alk30gr", "alk30kat", "alkkon", "alkrisk",  # "alk_kat"
    "vx210_2",

    # nicotine
    "typ_ezig", "typ_hnb", "fag_sch", "hnb_fag_sch", "psecdi_sch",  # "Anzahl",

    # other_drugs
    "hasch_12", "ecsta_12", "lsd_12", # "heroi_12",
    "andop_12", "kokai_12", "crack_12", "pilz_12", # "nps_12",
    "amp_12", "meth_12", "fr12_he",

    # medication
    "m_schm", "m_schl", "m_ber", "m_anr", "m_app", "m_depr", "m_neur", "m_anab", "t_schm", "t_schl", "t_ber", "t_anr",
    "t_app", "t_depr", "t_neur", "t_anab",  # "pomi",
]  # fmt: skip
corr_assoc_table[~corr_assoc_table["vari"].isin(excluded_vars) & ~corr_assoc_table["varj"].isin(excluded_vars)]

### alcohol variables

In [ ]:
# Look at variable group
vars_info[vars_info["group"].eq("alcohol")]

In [ ]:
# Ordinal alcohol variables
with pd.option_context("display.max_colwidth", 1000):
    display(vars_info[vars_info["name"].isin(["alk30kat", "alkkons", "alkkon", "alkrisk", "alk_kat"])])

In [ ]:
# Check if categorical variables are the same
var_list = ["alk30kat", "alkkons", "alkkon", "alkrisk", "alk_kat"]
display_side_by_side(
    *(
        contingency_table(data[vari], data[varj], dropna=False, normalize=False)
        for vari, varj in itertools.combinations(var_list, r=2)
    )
)

In [ ]:
# Actual values in the dataset
display_side_by_side(
    *(
        pd.DataFrame(data[var].value_counts(dropna=False, sort=False))
        for var in ["alk30kat", "alkkons", "alkkon", "alkrisk", "alk_kat"]
    )
)

### nicotine variables

In [ ]:
# Look at variable group
vars_info[vars_info["group"].eq("nicotine")]

In [ ]:
# Check if binary variables are the same
display_side_by_side(
    contingency_table(data["ezp30"], data["typ_ezig"], dropna=False, normalize=False),
    contingency_table(data["hnb30"], data["typ_hnb"], dropna=False, normalize=False),
)

### medication variables

In [ ]:
# Look at variable group
vars_info[vars_info["group"].eq("medication")]

In [ ]:
# Look at variable group
vars_info[(vars_info["group"].eq("medication")) & ~vars_info["name"].str.contains("_30|^m_|^t_", regex=True)]

### other_drugs variables

In [ ]:
# Look at variable group
vars_info[vars_info["group"].eq("other_drugs")]

In [ ]:
# Check if categorical variables are the same
vars_12 = [
    "hasch_12", "ecsta_12", "lsd_12", "heroi_12", "andop_12", "kokai_12", "crack_12", "pilz_12", "nps_12", "amp_12",
    "meth_12",
]  # fmt: skip
vars_fr12 = [
    "fr12_ha1", "fr12_ec1", "fr12_ls1", "fr12_he", "fr12_ao1", "fr12_ko1", "fr12_cr1", "fr12_pi1", "fr12_nps1",
    "fr12_au1", "fr12_meth1",
]  # fmt: skip
display_side_by_side(
    *(
        contingency_table(data[vari], data[varj], dropna=False, normalize=False)
        for vari, varj in zip(vars_12, vars_fr12)
    )
)

#### general, socioeconomic, and miscellaneous variables

In [ ]:
vars_info[~vars_info["group"].isin(["alcohol", "nicotine", "medication", "other_drugs"])]